### openai/whisper-large-v3-turbo  모델을 사용하여 분할된 음성에서 text 추출하기

모델 정보

* 모델: Whisper (예: large-v3 / large-v3-turbo)
* 모델 크기: 수억~수십억 파라미터 (선택 모델에 따라 다름)
* 특징:
    * 다국어 음성 인식 (한국어 포함)
    * 음성 → 텍스트 end-to-end 모델
    * 번역(Transcribe / Translate) 지원
* 출력:
    * 텍스트
    * 타임스탬프 (옵션)
* 한계:
    * 기본적으로 화자 구분(speaker diarization) 미지원
    * 긴 오디오에서는 chunk 처리 필요

⸻

Whisper 동작 원리 (핵심 이해)

Whisper는 “사람”이 아니라 **“언어적 음성 패턴”**을 인식합니다.

즉:

* ✔ 인간 음성 → 매우 잘 인식
* ✔ AI/TTS 음성 → 대부분 인식 가능
* ✔ 방송/유튜브 음성 → 안정적
* ⚠️ 노래 → 일부 가능 (불안정)
* ❌ 동물 소리/효과음 → 거의 불가

핵심 흐름:

오디오 → 특징 추출 → 언어 패턴 해석 → 텍스트 생성

⸻

수행 과제

이 노트북에서는 Whisper를 활용하여
다양한 음성 조건에서의 인식 성능을 비교 실험합니다.

⸻

구성:

Part A – Short Audio Test (숏폼 음성 인식)

* 짧은 음성 (예: 유튜브 Shorts, 짧은 클립)
* 빠른 응답 속도 및 기본 인식 정확도 확인

⸻

Part B – Multilingual Audio Test (다국어 음성 테스트)

* 한국어 + 영어 혼용 인터뷰
* 코드 스위칭 상황에서의 인식 성능 평가
* 언어 전환 시 오류 패턴 분석

예:

* 한국어 문장 중 영어 단어 삽입
* 영어 인터뷰 중 한국어 답변

⸻

Part C – Non-Human / Edge Case Test (특수 음성 실험)

* 앵무새/동물의 인간 발화 모방

목표:

* Whisper가 “언어”로 인식하는 기준 이해

⸻

학습 포인트

이 실습을 통해 다음을 배울 수 있습니다:

* 음성 인식 모델의 작동 원리 이해
* 긴 오디오 처리 시 필요한 Chunking 전략
* 다국어 및 혼합 언어 상황에서의 모델 한계

In [ ]:
# ----------------------------
# 설치
# ----------------------------
!pip install --upgrade pip
!pip install --upgrade transformers datasets[audio] accelerate

In [ ]:
# ----------------------------
# 1. 모델 로드 및 제공된 예시 테스트
# ----------------------------

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float32 # Changed from torch.float16 to torch.float32 for debugging

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

# Pass return_timestamps=True to handle long audio inputs
result = pipe(sample, return_timestamps=True)
print(result["text"])

In [ ]:
# ----------------------------
# 2. 유튜브 쇼츠 테스트
# ----------------------------
# 앵무새 루이 > 짧은 영상 11초 > 자존감 높은 앵무새 김루이
# https://youtube.com/shorts/fthxEWthtzU


result = pipe("자존감 높은 앵무새 김루이.mp3", return_timestamps=True)
#print(result["text"])

for chunk in result["chunks"]:
    print(chunk["timestamp"], chunk["text"])

(0.0, 3.6)  루이야, 너는 왜 이렇게 뚱뚱해?
(4.1, 6.1)  나는 멋있게, 나는 멋있게
(6.1, 7.34)  나는 멋있게
(7.34, 8.78)  너는 멋있게, 니가 멋있어?
(9.26, 11.1)  너 뚱뚱뚱해


In [ ]:
# ----------------------------
# 3. 유튜브 영상 테스트
# ----------------------------
# 앵무새 루이 > 영상 4분 30초 > 각자 다른 아무 말 대잔치
# https://youtu.be/QTZ8E5MyS8M

result = pipe("각자 다른 아무 말 대잔치 #녹취록.mp3", return_timestamps=True)

for chunk in result["chunks"]:
    print(chunk["timestamp"], chunk["text"])

(0.0, 6.0)  내가 30만원에 남파야, 이만하오, 남파야, 내가 30만원에 남파야, 남파야, 내가 30만원에 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 남파야, 아이고, 아이고, 기념을 못 표현할거에요.
(6.0, 9.0)  낮은 어빙수가 없다.
(9.0, 12.0)  그 팔 때, 잠기 쉽다고 해서
(12.0, 15.0)  아이고, 아이고, 그래서
(23.0, 25.0)  낮은 어빙수가 없다.
(25.0, 27.0)  속고 있는 속고야.
(27.0, 29.0)  이리와봐, 이리와봐
(31.0, 33.0)  왜 이리와봐
(35.0, 37.0)  왜 이리와봐
(37.0, 39.0)  왜 이리와봐
(43.0, 45.0)  왜 이리와봐
(45.0, 47.0)  왜 이리와봐
(47.0, 49.0)  왜 이리와봐
(49.0, 51.0)  왜 이리와봐
(51.0, 53.0)  왜 이리와봐
(53.0, 55.0)  왜 이리와봐
(55.0, 57.0)  한글자막 by 한글자막 by 한글자막 by 한글자막 by 한글자막 by 

In [ ]:
# ----------------------------
# 4. 유튜브 긴 영상 테스트
# ----------------------------
# 긴영상 27분 30초 > 스페셜 인터뷰 TEAM 프로젝트 헤일메리 | 라플위클리 HOT & NEW ep.02 - 프로젝트 헤일메리
# https://youtu.be/Arc0hWf06Qk

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=20,
    batch_size=4,
    torch_dtype=torch_dtype,
    device=device,
)

result = pipe("스페셜 인터뷰 TEAM 프로젝트 헤일메리.mp3", return_timestamps=True)
for chunk in result["chunks"]:
    print(chunk["timestamp"], chunk["text"])

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


(0.0, 5.12)  어쩌면 올해 최고의 SF 기대작
(5.12, 8.04)  전 세계가 기다려온 우주 블록버스터
(8.04, 9.78)  프로젝트 헬 메리
(9.78, 12.24)  주연 라이언 고슬린과의 만남
(12.24, 15.72)  로키의 오디션 현장부터
(15.72, 25.79)  로키의 오디션 현장부터 두 감독, 원작 작가, 각 본가에게 직접 들은 비하인드 썰까지
(25.79, 28.01)  준비되셨죠? 질문?
(32.85, 44.0)  질문 준비 되셨죠? 질문? 나풀 위클리 핫 앤 뉴 2회차 만에 초초특급 스페셜 게스트를 모실 수 있게 돼서 저희 지금 굉장히 분주하게
(44.0, 47.0)  스튜디오를 정리하고 있습니다.
(47.0, 50.0)  이분들 만날 준비해 보겠습니다.
(61.33, 67.33)  안녕하세요, 앤디. 안녕하세요. My name is Hyunmo and I'm from Life Plus TV in Korea. Annyeonghaseyo. I love you. That's all I know. That's great.
(67.33, 73.27)  Just in case you might be wondering, this your the korean copy of your book oh nice
(73.27, 80.95)  very nice what does the title translate to project hail mary oh literally okay cool yes
(80.95, 84.0)  first of all congratulations on the opening of the movie.
(84.0, 85.0)  Thank you.
(85.0, 89.0)  I know you were on the set a lot and you've been working closely with the cast and the
(89.0, 92.0)  crew, but how did you really like the fin